# IoT Telemetry Visual Profiling — Enterprise Edition

**Objective**: Detect sensor drift, outliers, and data quality issues through high-fidelity visual inspection.

---

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from src.iot.extract_iot import extract_iot
from src.iot.transform_iot import transform_iot_telemetry
from src.iot.quality_ge_iot import validate_iot_telemetry

# Set plotting style for premium feel
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12

## 1. Load Inbound Data
We extract data from the landing zone (JSON + CSV) using our modular **Enterprise Extractor**. This mimics pulling from a raw data lake.

In [ ]:
raw_df = extract_iot()
print(f"🔹 Raw data points retrieved: {len(raw_df)}")
raw_df.head()

## 2. Global Quality Profiling
Before filtering, we quantify the **'Chaos'** in the raw batch to understand the health of our sensor network.

In [ ]:
# Standardize for profiling
df = transform_iot_telemetry(raw_df)

# Summary: Missing Timestamps
missing_ts = df['reading_ts'].isna().sum()

# Summary: Unit Mismatches
unit_stats = df.groupby(['metric', 'unit']).size().reset_index(name='count')

# Summary: Outliers (Based on Hard Physical Bounds)
bounds = {
    'temp_c': (-40, 85),
    'humidity_pct': (0, 100),
    'pressure_hpa': (300, 1100)
}

outlier_counts = []
for metric, (low, high) in bounds.items():
    m_df = df[df['metric'] == metric]
    count = m_df[(m_df['value'] < low) | (m_df['value'] > high)].shape[0]
    outlier_counts.append({'metric': metric, 'outlier_count': count, 'min_bound': low, 'max_bound': high})

print(f"🕒 Missing Timestamps: {missing_ts}")
print("\n🌍 Unit Distribution (Governance Check):")
print(unit_stats)
print("\n🚫 Outlier Baseline (Integrity Check):")
display(pd.DataFrame(outlier_counts))

## 3. Deep-Dive Time-Series Analysis
We plot a single device feed to detect **sensor drift** and highlight anomalous points that our pipeline will block.

In [ ]:
target_device = df['device_id'].iloc[0]
metric = 'temp_c'
low, high = bounds[metric]

dev_df = df[(df['device_id'] == target_device) & (df['metric'] == metric)].sort_values('reading_ts')

# Separate clean and outliers for visualization
clean = dev_df[(dev_df['value'] >= low) & (dev_df['value'] <= high)]
outliers = dev_df[(dev_df['value'] < low) | (dev_df['value'] > high)]

plt.plot(clean['reading_ts'], clean['value'], label='Valid Signal', color='#2ecc71', linewidth=2, alpha=0.8)
if not outliers.empty:
    plt.scatter(outliers['reading_ts'], outliers['value'], label='Anomalous Reading (Quarantined)', color='#e74c3c', s=80, marker='x', zorder=5)

plt.axhline(y=high, color='#f39c12', linestyle='--', label='Critical Threshold (Upper)', alpha=0.6)
plt.axhline(y=low, color='#f39c12', linestyle='--', label='Critical Threshold (Lower)', alpha=0.6)

plt.title(f"Real-Time Sensor Integrity: {target_device} | Metric: {metric}", pad=20)
plt.xlabel("Observation Timestamp")
plt.ylabel(f"Engineered Value ({metric})")
plt.legend(frameon=True, loc='best')
plt.xticks(rotation=30)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Run Pipeline Gate (Simulation)
Demonstrate how our **Great Expectations (GE)** logic programmatically enforces these bounds without manual intervention.

In [ ]:
print("🛡️ Executing Enterprise Quality Gate...")
results = validate_iot_telemetry(df)

if not results['success']:
    print("❌ GATE FAILED: Physical reality violated.")
    for exc in results['results']:
        if not exc['success']:
            print(f"  - [VIOLATION] Column: {exc['expectation_config']['kwargs'].get('column')} | Expectation: {exc['expectation_config']['expectation_type']}")
else:
    print("✅ GATE PASSED: Batch verified to match physical specifications.")

---

**End of Profiling**: The data is ready for the Silver layer (Loading into Postgres).